# CEVAE診断：data_seed=3 の失敗原因を model_seed で探る

10 seed 実験で seed=3, 6 だけ √PEHE が 7〜8 に跳ね上がった。
z_corr も同時に低下しており、**モデルが悪い局所解に収束している**可能性が高い。

このノートブックは **data_seed=3 だけに絞った診断専用ファイル**。
上から順に全部実行するだけで完結する（重い処理なし、約3分）。

ベスト設定（z_dim=1, hidden=32, beta_kl_max=1.0, weight_decay=1e-4）を固定したまま、
**model_seed（モデル初期値）だけを 0〜19 で変えて**、
初期値次第で失敗が救えるかを確認する。


In [ ]:
import warnings
import itertools
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy.stats import uniform, randint
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from xgboost import XGBRegressor
import torch
import torch.nn as nn

# macOS の Accelerate BLAS バックエンドが小さい行列の matmul で出す偽の警告を抑制
# （NaN/Inf が実際に出ていないことは確認済み。NumPy+Apple Silicon の既知の挙動）
warnings.filterwarnings("ignore", message=".*encountered in matmul.*", category=RuntimeWarning)


## 1. データ生成（data_seed=3 固定）

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def generate_toy_v2(N=20000, d=8, gamma=1.5, theta=2.0, alpha=1.0, beta=1.0, delta_nl=0.8,
                     sigma_x=3.5, sigma_y=1.0, link_scale=1.5, loading_mean=1.0, loading_std=1.0, seed=0):
    """v2トイデータ生成。Z→Xに非線形リンク tanh、tau に非線形項(Z²-1)を追加。"""
    rng = np.random.default_rng(seed)
    Z = rng.normal(0, 1, size=N)
    g_z = np.tanh(link_scale * Z)
    a = rng.normal(loading_mean, loading_std, size=d)
    X = g_z[:, None] * a[None, :] + rng.normal(0, sigma_x, size=(N, d))
    T = rng.binomial(1, sigmoid(gamma * Z))
    tau = alpha + beta * Z + delta_nl * (Z ** 2 - 1)
    base = theta * Z
    Y0 = base + rng.normal(0, sigma_y, size=N)
    Y1 = base + tau + rng.normal(0, sigma_y, size=N)
    Y = np.where(T == 1, Y1, Y0)
    return X, T, Y, Y0, Y1, tau, Z


X, T, Y, Y0, Y1, tau, Z = generate_toy_v2()
d = X.shape[1]

# 診断
true_ate = tau.mean()
naive_ate = Y[T == 1].mean() - Y[T == 0].mean()
corr_t_z = np.corrcoef(T, Z)[0, 1]
Xd = np.column_stack([np.ones(len(Z)), X])
coef, _, _, _ = np.linalg.lstsq(Xd, Z, rcond=None)
r2_zx = 1 - np.sum((Z - Xd @ coef) ** 2) / np.sum((Z - Z.mean()) ** 2)
corr_tau_z = np.corrcoef(tau, Z)[0, 1]
corr_tau_z2 = np.corrcoef(tau, Z ** 2)[0, 1]

print(f"N={len(Z)}, d={d}")
print(f"真のATE           : {true_ate:.3f}  (≈ alpha=1.0)")
print(f"ナイーブATE       : {naive_ate:.3f}  バイアス={naive_ate-true_ate:+.3f}")
print(f"corr(T, Z)        : {corr_t_z:.3f}  (交絡の強さ)")
print(f"R²(Z|X) 線形回帰  : {r2_zx:.3f}  (元のトイデータは0.81 → 大幅に低下しているはず)")
print(f"corr(tau, Z)      : {corr_tau_z:.3f}  (線形成分)")
print(f"corr(tau, Z²)     : {corr_tau_z2:.3f}  (非線形成分. 0でなければ非線形性がある)")


# data_seed=3 でデータ生成・分割
DATA_SEED = 3
X, T, Y, Y0, Y1, tau, Z = generate_toy_v2(seed=DATA_SEED)
d = X.shape[1]
proxy_cols = [f"X{j+1}" for j in range(d)]

df = pd.DataFrame(X, columns=proxy_cols)
df["T"] = T; df["Y"] = Y; df["tau"] = tau

df_train, df_test = train_test_split(df, test_size=0.3, random_state=DATA_SEED, stratify=df["T"])

x_train = torch.tensor(df_train[proxy_cols].values, dtype=torch.float32)
t_train = torch.tensor(df_train["T"].values,        dtype=torch.float32)
y_train = torch.tensor(df_train["Y"].values,        dtype=torch.float32)
x_test  = torch.tensor(df_test[proxy_cols].values,  dtype=torch.float32)
Z_train = pd.Series(Z, index=df.index).loc[df_train.index].to_numpy()
tau_test = df_test["tau"].values

print(f"data_seed={DATA_SEED}  train={len(df_train)}  test={len(df_test)}  d={d}")


## 2. ネットワーク・関数定義

In [ ]:
torch.manual_seed(0)

Z_DIM = 1     # CEVAEの潜在変数zの次元（真のZがスカラーなので、まずはz_dim=1に合わせる）
HIDDEN = 64   # デフォルト（1回目の実験での最良設定。build_cevae経由で変更可能）


class DecoderX(nn.Module):
    """p(x|z)：zからproxy X1~X5の平均を予測する（ガウス分布、分散は固定とみなす）"""
    def __init__(self, z_dim=Z_DIM, d=d, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, d),
        )

    def forward(self, z):
        return self.net(z)          # (batch, d) Xの平均


class DecoderT(nn.Module):
    """p(t|z)：zから処置Tのロジットを予測する（Bernoulli）"""
    def __init__(self, z_dim=Z_DIM, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, z):
        return self.net(z).squeeze(-1)   # (batch,) ロジット（sigmoidを通す前の値）


class DecoderY(nn.Module):
    """p(y|t,z)：TARNet構造。t=0用・t=1用で完全に別のヘッド（ネットワーク）を持つ"""
    def __init__(self, z_dim=Z_DIM, hidden=HIDDEN):
        super().__init__()
        self.head0 = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        self.head1 = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, z):
        y0_hat = self.head0(z).squeeze(-1)   # p(y|t=0, z) の平均
        y1_hat = self.head1(z).squeeze(-1)   # p(y|t=1, z) の平均
        return y0_hat, y1_hat


decoder_x = DecoderX()
decoder_t = DecoderT()
decoder_y = DecoderY()


class EncoderT(nn.Module):
    """q(t|x)：xから処置Tのロジットを予測する補助ネットワーク（テスト時にtを推定するために使う）"""
    def __init__(self, d=d, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)   # (batch,) ロジット


class EncoderY(nn.Module):
    """q(y|x,t)：TARNet構造の補助ネットワーク。t=0用・t=1用で別ヘッドの平均を出す
    （実際にどちらを使うかはtで選ぶ。テスト時にyを推定するために使う）"""
    def __init__(self, d=d, hidden=HIDDEN):
        super().__init__()
        self.head0 = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        self.head1 = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x):
        y0_hat = self.head0(x).squeeze(-1)   # q(y|x, t=0) の平均
        y1_hat = self.head1(x).squeeze(-1)   # q(y|x, t=1) の平均
        return y0_hat, y1_hat


class EncoderZ(nn.Module):
    """q(z|x,t,y)：本体の推論ネットワーク。
    まずxだけから共有表現hを作り（trunk）、tで枝分かれした先で[h, y]からzの平均・log分散を出す。
    実際に使う枝（t=0用 / t=1用）は観測されたtで選ぶ。
    """
    def __init__(self, d=d, z_dim=Z_DIM, hidden=HIDDEN):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(d, hidden), nn.ReLU())
        self.head0 = nn.Sequential(nn.Linear(hidden + 1, hidden), nn.ReLU())
        self.head1 = nn.Sequential(nn.Linear(hidden + 1, hidden), nn.ReLU())
        self.mu0 = nn.Linear(hidden, z_dim)
        self.logvar0 = nn.Linear(hidden, z_dim)
        self.mu1 = nn.Linear(hidden, z_dim)
        self.logvar1 = nn.Linear(hidden, z_dim)

    def forward(self, x, t, y):
        h = self.trunk(x)                             # (batch, hidden) ← xだけから作る共有表現
        hy = torch.cat([h, y.unsqueeze(-1)], dim=-1)   # (batch, hidden+1) ← yを連結
        h0 = self.head0(hy)
        h1 = self.head1(hy)
        mu0, logvar0 = self.mu0(h0), self.logvar0(h0)
        mu1, logvar1 = self.mu1(h1), self.logvar1(h1)
        t_col = t.unsqueeze(-1)                        # (batch, 1) ← (batch, z_dim)とブロードキャストするため
        mu = torch.where(t_col == 1, mu1, mu0)
        logvar = torch.where(t_col == 1, logvar1, logvar0)
        return mu, logvar


def reparameterize(mu, logvar):
    """reparameterization trick：z = mu + eps*std（epsは標準正規からサンプル）。
    勾配がサンプリングを通り抜けられるようにするためのCEVAE/VAEの定石。
    """
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std


encoder_t = EncoderT()
encoder_y = EncoderY()
encoder_z = EncoderZ()

def kl_to_standard_normal(mu, logvar):
    """q(z|x,t,y) = N(mu, exp(logvar)) と事前分布 N(0,I) のKLダイバージェンス（解析解）。
    batch平均を返す（z_dim方向はsumしてからbatch方向にmeanする）。
    """
    kl_per_dim = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())   # (batch, z_dim)
    return kl_per_dim.sum(dim=-1).mean()


def linear_kl_anneal(step: int, total_steps: int) -> float:
    """KLアニーリングの重み beta_kl を 0→1 へ線形に上げるスケジュール。
    学習ループ側で `beta_kl = linear_kl_anneal(現在のstep, アニーリング終了までのstep数)` のように使う想定。
    """
    if total_steps <= 0:
        return 1.0
    return min(1.0, step / total_steps)


def cevae_loss(x, t, y, beta_kl=1.0):
    """CEVAEの1バッチ分の損失（−ELBO + 補助損失）を計算する。
    x: (batch, d)  t: (batch,) 0/1  y: (batch,)
    decoder_x/decoder_t/decoder_y（Part 1）と encoder_t/encoder_y/encoder_z（Part 2）を使う。
    戻り値: (total_loss, 各項のdict（ログ用、勾配は持たないfloatに変換済み）)
    """
    # 1) 事後分布 q(z|x,t,y) から z をサンプル
    mu_z, logvar_z = encoder_z(x, t, y)
    z = reparameterize(mu_z, logvar_z)

    # 2) Decoder で x, t, y を再構成
    x_hat = decoder_x(z)
    t_logit_p = decoder_t(z)
    y0_hat_p, y1_hat_p = decoder_y(z)
    y_hat_p = torch.where(t == 1, y1_hat_p, y0_hat_p)

    recon_x = nn.functional.mse_loss(x_hat, x)
    recon_t = nn.functional.binary_cross_entropy_with_logits(t_logit_p, t)
    recon_y = nn.functional.mse_loss(y_hat_p, y)

    # 3) KL( q(z|x,t,y) || N(0,I) )
    kl = kl_to_standard_normal(mu_z, logvar_z)

    # 4) 補助損失：テスト時にxだけからt・yを当てるための補助ネットワークもここで学習させる
    t_logit_q = encoder_t(x)
    y0_hat_q, y1_hat_q = encoder_y(x)
    y_hat_q = torch.where(t == 1, y1_hat_q, y0_hat_q)

    aux_t = nn.functional.binary_cross_entropy_with_logits(t_logit_q, t)
    aux_y = nn.functional.mse_loss(y_hat_q, y)

    total = recon_x + recon_t + recon_y + beta_kl * kl + aux_t + aux_y

    components = {
        "total": total.item(),
        "recon_x": recon_x.item(),
        "recon_t": recon_t.item(),
        "recon_y": recon_y.item(),
        "kl": kl.item(),
        "aux_t": aux_t.item(),
        "aux_y": aux_y.item(),
    }
    return total, components

@dataclass
class CEVAEConfig:
    """CEVAEの構成・学習ハイパーパラメータ一式。Part1〜5で固定だった値を全部ここに集約する。"""
    z_dim: int = 1
    hidden: int = 32
    beta_kl_max: float = 1.0       # KLアニーリングが到達する最大の重み（Part1〜5は暗黙に1.0だった）
    kl_anneal_epochs: int = 75     # この epoch 数かけて beta_kl を 0→beta_kl_max へ上げる
    w_recon_y: float = 1.0         # 再構成損失のうちyの項の重み
    w_aux_y: float = 1.0           # 補助損失のうちyの項の重み
    epochs: int = 150
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 0.0      # Adamのweight decay（L2正則化）。0.0なら今までと同じ
    seed: int = 0


def build_cevae(cfg: CEVAEConfig, d: int) -> dict:
    """cfg.z_dim, cfg.hidden に従って、Decoder3つ・Encoder3つを毎回ゼロから新規作成する。
    Part1/Part2で定義したクラス（DecoderX/T/Y, EncoderT/Y/Z）はもともとz_dim/hiddenを
    引数で受け取れる作りなので、そのまま再利用できる。
    """
    torch.manual_seed(cfg.seed)
    return {
        "decoder_x": DecoderX(z_dim=cfg.z_dim, d=d, hidden=cfg.hidden),
        "decoder_t": DecoderT(z_dim=cfg.z_dim, hidden=cfg.hidden),
        "decoder_y": DecoderY(z_dim=cfg.z_dim, hidden=cfg.hidden),
        "encoder_t": EncoderT(d=d, hidden=cfg.hidden),
        "encoder_y": EncoderY(d=d, hidden=cfg.hidden),
        "encoder_z": EncoderZ(d=d, z_dim=cfg.z_dim, hidden=cfg.hidden),
    }


def cevae_loss_v2(nets: dict, x, t, y, beta_kl=1.0, w_recon_y=1.0, w_aux_y=1.0):
    """Part3の cevae_loss を一般化したもの。グローバル変数ではなく nets 辞書を受け取り、
    recon_y・aux_y の重みを外から指定できるようにした。中身のロジックはPart3と同じ。
    """
    mu_z, logvar_z = nets["encoder_z"](x, t, y)
    z = reparameterize(mu_z, logvar_z)

    x_hat = nets["decoder_x"](z)
    t_logit_p = nets["decoder_t"](z)
    y0_hat_p, y1_hat_p = nets["decoder_y"](z)
    y_hat_p = torch.where(t == 1, y1_hat_p, y0_hat_p)

    recon_x = nn.functional.mse_loss(x_hat, x)
    recon_t = nn.functional.binary_cross_entropy_with_logits(t_logit_p, t)
    recon_y = nn.functional.mse_loss(y_hat_p, y)

    kl = kl_to_standard_normal(mu_z, logvar_z)

    t_logit_q = nets["encoder_t"](x)
    y0_hat_q, y1_hat_q = nets["encoder_y"](x)
    y_hat_q = torch.where(t == 1, y1_hat_q, y0_hat_q)

    aux_t = nn.functional.binary_cross_entropy_with_logits(t_logit_q, t)
    aux_y = nn.functional.mse_loss(y_hat_q, y)

    total = recon_x + recon_t + w_recon_y * recon_y + beta_kl * kl + aux_t + w_aux_y * aux_y

    components = {
        "total": total.item(), "recon_x": recon_x.item(), "recon_t": recon_t.item(),
        "recon_y": recon_y.item(), "kl": kl.item(), "aux_t": aux_t.item(), "aux_y": aux_y.item(),
    }
    return total, components


def train_cevae(nets: dict, cfg: CEVAEConfig, x_tr, t_tr, y_tr, verbose_every: int = 0):
    """Part4の学習ループを一般化したもの。nets内の全パラメータをAdam1つでまとめて最適化する。
    戻り値: epochごとの損失内訳（list of dict）。verbose_every>0なら進捗を print する。
    """
    torch.manual_seed(cfg.seed)
    all_params = [p for net in nets.values() for p in net.parameters()]
    optimizer = torch.optim.Adam(all_params, lr=cfg.lr, weight_decay=cfg.weight_decay)

    n_samples = x_tr.shape[0]
    n_batches = max(1, n_samples // cfg.batch_size)
    history = []

    for epoch in range(cfg.epochs):
        beta_kl = cfg.beta_kl_max * linear_kl_anneal(epoch, cfg.kl_anneal_epochs)
        perm = torch.randperm(n_samples)
        epoch_sums = {"total": 0.0, "recon_x": 0.0, "recon_t": 0.0, "recon_y": 0.0, "kl": 0.0, "aux_t": 0.0, "aux_y": 0.0}

        for b in range(n_batches):
            idx = perm[b * cfg.batch_size:(b + 1) * cfg.batch_size]
            x_batch, t_batch, y_batch = x_tr[idx], t_tr[idx], y_tr[idx]

            optimizer.zero_grad()
            loss, components = cevae_loss_v2(nets, x_batch, t_batch, y_batch,
                                              beta_kl=beta_kl, w_recon_y=cfg.w_recon_y, w_aux_y=cfg.w_aux_y)
            loss.backward()
            optimizer.step()

            for k in epoch_sums:
                epoch_sums[k] += components[k]

        epoch_avg = {k: v / n_batches for k, v in epoch_sums.items()}
        epoch_avg["beta_kl"] = beta_kl
        history.append(epoch_avg)

        if verbose_every and (epoch % verbose_every == 0 or epoch == cfg.epochs - 1):
            c = epoch_avg
            print(f"epoch {epoch:4d}  beta_kl={c['beta_kl']:.2f}  total={c['total']:.3f}  "
                  f"recon_x={c['recon_x']:.3f} recon_t={c['recon_t']:.3f} recon_y={c['recon_y']:.3f} "
                  f"kl={c['kl']:.3f} aux_t={c['aux_t']:.3f} aux_y={c['aux_y']:.3f}")

    return history


def estimate_ite_v2(nets: dict, x):
    """Part5の estimate_ite を一般化したもの（nets辞書を受け取る版）。手順は設計書§4.4のまま。"""
    with torch.no_grad():
        t_prob = torch.sigmoid(nets["encoder_t"](x))
        t_hat = (t_prob > 0.5).float()

        y0_hat_q, y1_hat_q = nets["encoder_y"](x)
        y_hat = torch.where(t_hat == 1, y1_hat_q, y0_hat_q)

        mu_z, _ = nets["encoder_z"](x, t_hat, y_hat)

        y0_p, y1_p = nets["decoder_y"](mu_z)
        ite_pred = y1_p - y0_p
    return ite_pred


def diagnose_z_recovery(nets: dict, x, t, y, z_true) -> dict:
    """診断1を一般化したもの。encoder_z(x,t,y)のmu_zと真のZの相関を、符号反転を考慮して計算する。
    z_dimが2以上の場合は次元ごとに見て、絶対値最大のものを「z復元スコア」として返す
    （複数次元のうち1本だけが真のZ方向を捉え、他はノイズ、ということがあり得るため）。
    """
    with torch.no_grad():
        mu_z, _ = nets["encoder_z"](x, t, y)
    mu_z_np = mu_z.numpy()

    per_dim = []
    best_abs_corr = 0.0
    for k in range(mu_z_np.shape[1]):
        corr_pos = np.corrcoef(mu_z_np[:, k], z_true)[0, 1]
        per_dim.append((k, corr_pos))
        best_abs_corr = max(best_abs_corr, abs(corr_pos))

    return {"per_dim": per_dim, "best_abs_corr": best_abs_corr}


def evaluate_cevae(nets: dict, x_test, tau_true_test) -> dict:
    """テストセットでの ITE評価（√PEHE/MAE/eps_ATE）を返す。z復元度は別途 diagnose_z_recovery を使う。
    Part7のグリッドサーチで1設定ぶんの結果として使う。
    """
    tau_pred = estimate_ite_v2(nets, x_test).numpy()
    sqrt_pehe_v = np.sqrt(np.mean((tau_true_test - tau_pred) ** 2))
    mae_v = np.mean(np.abs(tau_true_test - tau_pred))
    eps_ate_v = abs(np.mean(tau_pred) - np.mean(tau_true_test))
    return {
        "sqrt_pehe": sqrt_pehe_v, "mae": mae_v, "eps_ate": eps_ate_v, "tau_pred": tau_pred,
    }

## 3. model_seed 0〜19 で試す

**固定する設定**（グリッドサーチ・WD実験の結果から）：
- z_dim=1, hidden=32, beta_kl_max=1.0, kl_anneal_epochs=75, epochs=150
- weight_decay=1e-4（WD=1e-4 で 8/10 seed が改善していたため）

**変えるもの**：model_seed（モデルの重み初期値・ミニバッチ順）のみ

**見たいこと**：
- z_corr が 0.9 以上になる model_seed はあるか？
- そのとき √PEHE も改善するか？（z_corr ↑ → √PEHE ↓ の対応があれば「初期値次第で救える」）


In [ ]:
BEST_CFG = dict(
    z_dim=1, hidden=32, beta_kl_max=1.0,
    kl_anneal_epochs=75, epochs=150,
    lr=1e-3, weight_decay=1e-4,
)
N_MODEL_SEEDS = 20

records = []
for model_seed in range(N_MODEL_SEEDS):
    cfg = CEVAEConfig(**BEST_CFG, seed=model_seed)
    nets = build_cevae(cfg, d=d)
    train_cevae(nets, cfg, x_train, t_train, y_train, verbose_every=0)

    ev = evaluate_cevae(nets, x_test, tau_test)
    dg = diagnose_z_recovery(nets, x_train, t_train, y_train, Z_train)

    records.append({
        "model_seed": model_seed,
        "sqrt_pehe":  ev["sqrt_pehe"],
        "eps_ate":    ev["eps_ate"],
        "z_corr":     dg["best_abs_corr"],
    })

    status = "✅" if ev["sqrt_pehe"] < 2.5 else "❌ 失敗"
    print(f"model_seed={model_seed:2d}  √PEHE={ev['sqrt_pehe']:.3f}  "
          f"z_corr={dg['best_abs_corr']:.3f}  {status}")

df_seeds = pd.DataFrame(records)


In [ ]:
print(f"\n成功(√PEHE<2.5): {(df_seeds['sqrt_pehe']<2.5).sum()}/{N_MODEL_SEEDS} seeds")
print(f"失敗(√PEHE>=2.5): {(df_seeds['sqrt_pehe']>=2.5).sum()}/{N_MODEL_SEEDS} seeds")
print(f"\nz_corr と √PEHE の相関: {df_seeds['z_corr'].corr(df_seeds['sqrt_pehe']):.3f}")
print("  （-1に近いほど z_corr↑ → √PEHE↓ の対応が強い）")
print()
print("全結果（√PEHE 昇順）:")
df_seeds.sort_values("sqrt_pehe").reset_index(drop=True)


## 4. CEVAE内部の y 予測比較：q(y|x,t) vs test-time p(y|z,t)

**問い**：zを挟むことで y の説明力が上がっているのか？

| | 説明 |
|---|---|
| `q(y|x,t)` | encoder_y が x と observed t から直接 y を予測（z なし） |
| `p(y|z,t)` test-time | test-time と同じ流れで z を推定してから y を予測 |

test-time の z 推定フロー（真の y を使わない）：
```
encoder_t(x) → t_hat
encoder_y(x)[t_hat] → y_hat_q   ← 真の y の代わりに使う
encoder_z(x, t_hat, y_hat_q) → z_mu
decoder_y(z_mu)[observed t] → y_p_hat
```

**読み方**：
- `mse_diff = mse_p - mse_q` が **負 → zを挟むと y 予測が改善**（z 経由が有効）
- `mse_diff` が **正 → zを挟んでも改善しない**（q(y|x,t) の方が直接的に強い）
- 成功seed(z_corr≈0.94) と 失敗seed(z_corr≈0.47) でこの値がどう変わるかを見る


In [ ]:
import matplotlib.pyplot as plt

# ─── y 予測比較の計算関数 ───────────────────────────────────────────
def compute_y_mse(nets, x_te, t_te_obs, y_te):
    """
    q(y|x,t) と test-time p(y|z,t) のMSEを計算する。

    Parameters
    ----------
    x_te      : test の x テンソル
    t_te_obs  : test の observed t テンソル（MSEの計算・ヘッド選択に使う）
    y_te      : test の true y （numpy）
    """
    with torch.no_grad():
        # ── q(y|x,t)：encoder_y で observed t のヘッドを選んで直接予測 ──
        y0_q, y1_q = nets["encoder_y"](x_te)
        y_q_hat = torch.where(t_te_obs == 1, y1_q, y0_q)

        # ── test-time p(y|z,t)：真の y を使わずに z を推定してから予測 ──
        # step1: q(t|x) で t_hat を推定
        t_hat = (torch.sigmoid(nets["encoder_t"](x_te)) > 0.5).float()
        # step2: q(y|x, t_hat) で y_hat_q を作る（z 推定に使う。真の y の代替）
        y0_for_z, y1_for_z = nets["encoder_y"](x_te)
        y_hat_for_z = torch.where(t_hat == 1, y1_for_z, y0_for_z)
        # step3: encoder_z(x, t_hat, y_hat_q) で z を推定
        z_mu, _ = nets["encoder_z"](x_te, t_hat, y_hat_for_z)
        # step4: decoder_y(z_mu) で p(y|z,t) を予測（observed t でヘッド選択）
        y0_p, y1_p = nets["decoder_y"](z_mu)
        y_p_hat = torch.where(t_te_obs == 1, y1_p, y0_p)

    y_q = y_q_hat.numpy()
    y_p = y_p_hat.numpy()
    t_np = t_te_obs.numpy()
    m1, m0 = t_np == 1, t_np == 0

    def mse(a, b): return float(np.mean((a - b) ** 2))

    mse_q   = mse(y_te, y_q)
    mse_p   = mse(y_te, y_p)
    return {
        "mse_q_y":             mse_q,
        "mse_p_y_testtime":    mse_p,
        "mse_q_y_t1":          mse(y_te[m1], y_q[m1]),
        "mse_q_y_t0":          mse(y_te[m0], y_q[m0]),
        "mse_p_y_testtime_t1": mse(y_te[m1], y_p[m1]),
        "mse_p_y_testtime_t0": mse(y_te[m0], y_p[m0]),
        "mse_diff":            mse_p - mse_q,   # 負なら z 経由で改善
    }


# ─── 20 model_seed で再学習しながら y 比較指標を取得 ──────────────────
y_te_tensor = torch.tensor(df_test["Y"].values, dtype=torch.float32)
t_te_tensor = torch.tensor(df_test["T"].values, dtype=torch.float32)
y_te_np     = df_test["Y"].values

records_y = []
for model_seed in range(N_MODEL_SEEDS):
    cfg  = CEVAEConfig(**BEST_CFG, seed=model_seed)
    nets = build_cevae(cfg, d=d)
    train_cevae(nets, cfg, x_train, t_train, y_train, verbose_every=0)

    ev   = evaluate_cevae(nets, x_test, tau_test)
    dg   = diagnose_z_recovery(nets, x_train, t_train, y_train, Z_train)
    ymse = compute_y_mse(nets, x_test, t_te_tensor, y_te_np)

    row = {
        "model_seed": model_seed,
        "sqrt_pehe":  ev["sqrt_pehe"],
        "eps_ate":    ev["eps_ate"],
        "z_corr":     dg["best_abs_corr"],
        **ymse,
    }
    records_y.append(row)

    status = "✅" if ev["sqrt_pehe"] < 2.5 else "❌"
    sign   = "↓改善" if ymse["mse_diff"] < 0 else "↑悪化"
    print(f"seed={model_seed:2d}  √PEHE={ev['sqrt_pehe']:.3f}  z_corr={dg['best_abs_corr']:.3f}  "
          f"mse_q={ymse['mse_q_y']:.3f}  mse_p={ymse['mse_p_y_testtime']:.3f}  "
          f"diff={ymse['mse_diff']:+.3f}({sign})  {status}")

df_y = pd.DataFrame(records_y)


# ─── 集計 ─────────────────────────────────────────────────────────
print("\n" + "="*70)
cols_disp = ["model_seed","sqrt_pehe","z_corr",
             "mse_q_y","mse_p_y_testtime","mse_diff",
             "mse_q_y_t1","mse_q_y_t0","mse_p_y_testtime_t1","mse_p_y_testtime_t0"]

print("\n全結果（model_seed 順）")
display(df_y[cols_disp].round(3))

print("\n平均・中央値")
summary = df_y[["sqrt_pehe","z_corr","mse_q_y","mse_p_y_testtime","mse_diff"]].agg(["mean","median"]).round(3)
display(summary)

success = df_y[df_y["sqrt_pehe"] < 2.5]
failure = df_y[df_y["sqrt_pehe"] >= 2.5]
print(f"\n成功seed({len(success)}個) vs 失敗seed({len(failure)}個) の比較:")
cmp_cols = ["z_corr","mse_q_y","mse_p_y_testtime","mse_diff"]
cmp = pd.DataFrame({
    "成功(mean)": success[cmp_cols].mean(),
    "失敗(mean)": failure[cmp_cols].mean(),
}).round(3)
display(cmp)


# ─── 棒グラフ ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

seeds = df_y["model_seed"]
x_pos = np.arange(len(seeds))
w = 0.35
colors_q = ["#4C72B0"] * len(seeds)
colors_p = ["#DD8452"] * len(seeds)
fail_mask = df_y["sqrt_pehe"] >= 2.5

ax = axes[0]
bars_q = ax.bar(x_pos - w/2, df_y["mse_q_y"],         w, label="q(y|x,t)", color=colors_q, alpha=0.85)
bars_p = ax.bar(x_pos + w/2, df_y["mse_p_y_testtime"], w, label="p(y|z,t) test-time", color=colors_p, alpha=0.85)
for i, fail in enumerate(fail_mask):
    if fail:
        ax.get_children()[i].set_edgecolor("red")
        ax.get_children()[i].set_linewidth(2)
        # annotate bars with red border for failed seeds
        ax.bar(x_pos[i] - w/2, df_y["mse_q_y"].iloc[i], w, color=colors_q[i], alpha=0.85, edgecolor="red", linewidth=2)
        ax.bar(x_pos[i] + w/2, df_y["mse_p_y_testtime"].iloc[i], w, color=colors_p[i], alpha=0.85, edgecolor="red", linewidth=2)
ax.set_xlabel("model_seed")
ax.set_ylabel("MSE")
ax.set_title("y予測MSE比較（赤枠=失敗seed）")
ax.set_xticks(x_pos); ax.set_xticklabels(seeds)
ax.legend()
ax.set_ylim(0, min(df_y[["mse_q_y","mse_p_y_testtime"]].max().max() * 1.2, 20))

ax2 = axes[1]
diff_colors = ["#2ecc71" if v < 0 else "#e74c3c" for v in df_y["mse_diff"]]
ax2.bar(x_pos, df_y["mse_diff"], color=diff_colors, alpha=0.85)
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax2.set_xlabel("model_seed")
ax2.set_ylabel("mse_diff = mse_p - mse_q")
ax2.set_title("z を挟むと改善(緑↓) or 悪化(赤↑)")
ax2.set_xticks(x_pos); ax2.set_xticklabels(seeds)
# 失敗seedに赤枠
for i, fail in enumerate(fail_mask):
    if fail:
        ax2.get_children()[i].set_edgecolor("black")
        ax2.get_children()[i].set_linewidth(2.5)

plt.tight_layout()
plt.show()
